Libararies

In [3]:
import pandas as pd
import numpy as np

from pathlib import Path

from pulp import LpProblem, LpMaximize, LpVariable, lpSum, LpBinary, PULP_CBC_CMD, value

### File Import

In [4]:
working_directory = Path.cwd().parent.parent
print(working_directory)


#FIXME: consider not having these vals hardcoded or wrapping in a read function that
#  has the ability to do both hard coded or default most recent week

file_path_drivers = working_directory / "data" / "predictions" / "drivers" / "driver_predictions_2026.csv" 
file_path_constr = working_directory / "data" / "predictions" / "constructors" / "constructor_preds_2026.csv"

predicted_points_constr = pd.read_csv(file_path_constr)
predicted_points_drivers = pd.read_csv(file_path_drivers).drop(columns=["Unnamed: 0"])


/Users/jackguptill/Library/CloudStorage/OneDrive-Personal/Code/F1FantasyProject


In [5]:
predicted_points_drivers.head()

,year,race_num,driver,price,constructor,predicted_points
0,2026,1,VER,27.7,RBR,38.163860
1,2026,1,RUS,27.4,MER,24.508209
2,2026,1,NOR,27.2,MCL,16.815132
3,2026,1,PIA,25.5,MCL,16.833676
4,2026,1,ANT,23.2,MER,20.579742


In [6]:
predicted_points_constr.head()

,year,race_num,constructor,price,predicted_points
0,2026,1,mcl,28.9,53.242756
1,2026,1,mer,29.3,51.584534
2,2026,1,rbr,28.2,48.359364
3,2026,1,fer,23.3,40.582527
4,2026,1,rb,6.3,26.878194


### Variables for the Linear Programming Problem

In [108]:
def optimize_team(
    budget: float,
    n_drivers: int = 5,
    n_constructors: int = 2,
    max_drivers_per_team: int | None = None,
    use_drs: bool = False,
    drs_multiplier: float = 2.0,
    solver_msg: bool = False,
    require_driver_from_each_constructor: bool = False,
    min_drivers_per_selected_constructor: int = 1,
):
    """
    Optimize an F1 Fantasy lineup using predicted points and prices.

    Expected CSV formats:
      drivers_csv columns: driver, constructor, price, predicted_points
      constructors_csv columns: constructor, price, predicted_points

    Returns:
      (drivers_selected_df, constructors_selected_df, summary_dict)
    """

    drivers = predicted_points_drivers
    cons = predicted_points_constr

    # Basic validation
    needed_driver_cols = {"driver", "constructor", "price", "predicted_points"}
    needed_cons_cols = {"constructor", "price", "predicted_points"}
    if not needed_driver_cols.issubset(drivers.columns):
        raise ValueError(f"drivers_csv missing columns: {needed_driver_cols - set(drivers.columns)}")
    if not needed_cons_cols.issubset(cons.columns):
        raise ValueError(f"constructors_csv missing columns: {needed_cons_cols - set(cons.columns)}")


    drivers["team_key"] = drivers["constructor"].astype(str).str.strip().str.upper()
    cons["team_key"] = cons["constructor"].astype(str).str.strip().str.upper()

    # Coerce types
    drivers["price"] = pd.to_numeric(drivers["price"], errors="coerce")
    drivers["predicted_points"] = pd.to_numeric(drivers["predicted_points"], errors="coerce")
    cons["price"] = pd.to_numeric(cons["price"], errors="coerce")
    cons["predicted_points"] = pd.to_numeric(cons["predicted_points"], errors="coerce")

    # Drop rows with missing essentials
    drivers = drivers.dropna(subset=["driver", "constructor", "price", "predicted_points"]).reset_index(drop=True)
    cons = cons.dropna(subset=["constructor", "price", "predicted_points"]).reset_index(drop=True)

    # Indices
    d_idx = drivers.index.tolist()
    c_idx = cons.index.tolist()

    # Problem
    prob = LpProblem("F1FantasyOptimizer", LpMaximize)

    # Decision vars
    x_d = LpVariable.dicts("pick_driver", d_idx, cat=LpBinary)
    x_c = LpVariable.dicts("pick_constructor", c_idx, cat=LpBinary)

    # Optional DRS vars: choose exactly 1 DRS driver among selected drivers
    if use_drs:
        z_drs = LpVariable.dicts("drs_driver", d_idx, cat=LpBinary)
        # DRS can only be applied to a selected driver
        for i in d_idx:
            prob += z_drs[i] <= x_d[i]
        prob += lpSum(z_drs[i] for i in d_idx) == 1
    else:
        z_drs = None

    # Objective
    # Without DRS: sum predicted points for selected drivers + constructors
    # With DRS: add (multiplier-1)*points for the chosen DRS driver
    obj = (
        lpSum(drivers.loc[i, "predicted_points"] * x_d[i] for i in d_idx)
        + lpSum(cons.loc[j, "predicted_points"] * x_c[j] for j in c_idx)
    )
    if use_drs:
        obj += (drs_multiplier - 1.0) * lpSum(drivers.loc[i, "predicted_points"] * z_drs[i] for i in d_idx)

    prob += obj

    # Constraints
    # Roster sizes
    prob += lpSum(x_d[i] for i in d_idx) == n_drivers
    prob += lpSum(x_c[j] for j in c_idx) == n_constructors

    # Budget
    prob += (
        lpSum(drivers.loc[i, "price"] * x_d[i] for i in d_idx)
        + lpSum(cons.loc[j, "price"] * x_c[j] for j in c_idx)
        <= budget
    )

    # Optional: max drivers per constructor/team
    if max_drivers_per_team is not None:
        for team in drivers["constructor"].unique():
            team_driver_indices = drivers.index[drivers["constructor"] == team].tolist()
            prob += lpSum(x_d[i] for i in team_driver_indices) <= max_drivers_per_team
    
    if require_driver_from_each_constructor:
        for j in c_idx:
            team = cons.loc[j, "team_key"]
            team_driver_indices = drivers.index[drivers["team_key"] == team].tolist()

            # If your constructor list includes a team that has no drivers in the drivers df,
            # this would make the model infeasible. Guard it.
            if not team_driver_indices:
                # Either skip, or raise an error. I prefer raising so you catch data issues early.
                raise ValueError(f"No drivers found for constructor/team '{team}' in drivers df.")

            prob += lpSum(x_d[i] for i in team_driver_indices) >= min_drivers_per_selected_constructor * x_c[j]

    # Solve
    solver = PULP_CBC_CMD(msg=solver_msg)
    prob.solve(solver)

    # Extract results
    picked_driver_rows = [i for i in d_idx if value(x_d[i]) == 1]
    picked_cons_rows = [j for j in c_idx if value(x_c[j]) == 1]

    drivers_sel = drivers.loc[picked_driver_rows].copy().sort_values("predicted_points", ascending=False)
    cons_sel = cons.loc[picked_cons_rows].copy().sort_values("predicted_points", ascending=False)

    # DRS driver
    drs_driver = None
    if use_drs:
        drs_picks = [i for i in d_idx if value(z_drs[i]) == 1]
        if drs_picks:
            drs_driver = drivers.loc[drs_picks[0], "driver"]

    total_price = float(drivers_sel["price"].sum() + cons_sel["price"].sum())
    total_points = float(drivers_sel["predicted_points"].sum() + cons_sel["predicted_points"].sum())
    if use_drs and drs_driver is not None:
        drs_points = float(drivers_sel.loc[drivers_sel["driver"] == drs_driver, "predicted_points"].iloc[0])
        total_points += (drs_multiplier - 1.0) * drs_points

    summary = {
        "status": prob.status,  # 1 is optimal in PuLP; you can map if you want
        "budget": budget,
        "total_price": round(total_price, 2),
        "total_predicted_points": round(total_points, 2),
        "drs_driver": drs_driver,
    }

    return drivers_sel.reset_index(drop=True), cons_sel.reset_index(drop=True), summary

In [109]:
drivers_sel, cons_sel, summary = optimize_team(
    budget=100.0,
    n_drivers=5,
    n_constructors=2,
    max_drivers_per_team=2,   # optional
    use_drs=True,             # optional
    drs_multiplier=2.0,
    require_driver_from_each_constructor=True,
    min_drivers_per_selected_constructor=1,
)

print(summary)
display(drivers_sel)
display(cons_sel)

{'status': 1, 'budget': 100.0, 'total_price': 99.6, 'total_predicted_points': 189.61, 'drs_driver': 'VER'}


,driver,price,constructor,predicted_points,team_key
0,VER,27.7,RBR,38.163860,RBR
1,LEC,22.8,FER,21.135585,FER
2,HUL,6.8,AUD,9.227555,AUD
3,COL,6.2,ALP,8.773124,ALP
4,LAW,6.5,RB,6.684647,RB


,constructor,price,predicted_points,team_key
0,fer,23.3,40.582527,FER
1,rb,6.3,26.878194,RB
